# Backup Coverage Location Problem: An Empirical Example

*Authors:* [Erin Olson](https://github.com/erinrolson), [Germano Barcelos](https://github.com/gegen07), [James Gaboardi](https://github.com/jGaboardi), [Levi J. Wolf](https://github.com/ljwolf), [Qunshan Zhao](https://github.com/qszhao)

This tutorial extends the [Empirical examples](./facloc-real-world.ipynb) notebook, specifically for the Backup Coverage Location Set Covering Problem (LSCP-B). A deeper dive into the LSCP-B can be found [here](./lscpb.ipynb).
Also, this tutorial demonstrates the use of different solvers that [PULP](https://github.com/coin-or/pulp) supports.

In [ ]:
%config InlineBackend.figure_format = "retina"
%load_ext watermark
%watermark

In [ ]:
import time
import warnings
from pathlib import Path

import geopandas
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import matplotlib_scalebar
import pandas
import pulp
import shapely
from matplotlib.patches import Patch
from matplotlib_scalebar.scalebar import ScaleBar
from shapely.geometry import Point

import spopt
from spopt.locate import LSCP, LSCPB

%watermark -w
%watermark -iv

-------------------
We use 4 data files as input:

- `network_distance` is the distance between facility candidate sites calculated by ArcGIS Network Analyst Extension
- `demand_points` represents the demand points with some important features for the facility location problem like population
- `facility_points` represents the stores that are candidate facility sites
- `tract` is the polygon of census tract 205.

All datasets are online on this [repository](https://github.com/huanfachen/Open_source_location_cover_models/tree/master/Data/San_Francisco_store).

In [ ]:
DIRPATH = Path(spopt.__file__).parent.joinpath("tests/test_locate/data/")

`network_distance` dataframe

In [ ]:
network_distance = pandas.read_csv(
    DIRPATH.joinpath("SF_network_distance_candidateStore_16_censusTract_205_new.csv")
)
network_distance

`demand_points` dataframe

In [ ]:
demand_points = pandas.read_csv(
    DIRPATH.joinpath("SF_demand_205_centroid_uniform_weight.csv"), index_col=0
)
demand_points = demand_points.reset_index(drop=True)
demand_points

`facility_points` dataframe

In [ ]:
facility_points = pandas.read_csv(
    DIRPATH.joinpath("SF_store_site_16_longlat.csv"), index_col=0
)
facility_points = facility_points.reset_index(drop=True)
facility_points

`study_area` dataframe

In [ ]:
study_area = geopandas.read_file(DIRPATH.joinpath("ServiceAreas_4.shp")).dissolve()
study_area

Plot `study_area`

In [ ]:
base = study_area.plot()
base.axis("off");

To start modeling the problem assuming the arguments expected by `spopt.locate`, we should pass a `numpy` 2D array as a cost matrix. So, first we pivot the network_distance dataframe. 

_Note that the columns and rows are sorted in ascending order._

In [ ]:
ntw_dist_piv = network_distance.pivot_table(
    values="distance", index="DestinationName", columns="name"
)
ntw_dist_piv

Here the pivot table is transformed to numpy 2D array such as `spopt.locate` expected. The matrix has a shape of 205x16.

In [ ]:
cost_matrix = ntw_dist_piv.to_numpy()
cost_matrix.shape

In [ ]:
cost_matrix[:3, :3]

Now, as the rows and columns of our cost matrix are sorted, we have to sort our facility points and demand points geodataframes, too. 

In [ ]:
n_dem_pnts = demand_points.shape[0]
n_fac_pnts = facility_points.shape[0]

In [ ]:
process = lambda df: as_gdf(df).sort_values(by=["NAME"]).reset_index(drop=True)  # noqa: E731
as_gdf = lambda df: geopandas.GeoDataFrame(df, geometry=pnts(df))  # noqa: E731
pnts = lambda df: geopandas.points_from_xy(df.long, df.lat)  # noqa: E731

In [ ]:
facility_points = process(facility_points)
demand_points = process(demand_points)

Reproject the input spatial data.

In [ ]:
for _df in [facility_points, demand_points, study_area]:
    _df.set_crs("EPSG:4326", inplace=True)
    _df.to_crs("EPSG:7131", inplace=True)

Set parameter `SERVICE_RADIUS` for maximum service standard of distance or time.

In [ ]:
# maximum service radius (in meters)
SERVICE_RADIUS = 8000

In [ ]:
solver = pulp.PULP_CBC_CMD(msg=False)

## LSCP & LSCP-B considering all candidate facilities

In [ ]:
lscp = LSCP.from_cost_matrix(cost_matrix, SERVICE_RADIUS)
lscp = lscp.solve(solver)

In [ ]:
lscpb = LSCPB.from_cost_matrix(cost_matrix, SERVICE_RADIUS, solver)
lscpb = lscpb.solve()

In [ ]:
lscp_obj = lscp.problem.objective.value()
lscpb_obj = lscpb.problem.objective.value()
lscpb_lscp = lscpb.lscp_obj_value
lscpb_perc = round(lscpb.backup_perc, 3)
print(
    "The minimum number of facilites needed to achieve complete "
    f"coverage within a distance units service radius of {SERVICE_RADIUS} "
    f"are {lscp_obj} and {lscpb_lscp} for the LSCP and LSCP-B, "
    "respectively. However, with the LSCP-B we observe coverage by more "
    f"than one facility at {lscpb_perc}% of client locations."
)

Define the decision variable names used for mapping later.

In [ ]:
facility_points["dv"] = lscpb.fac_vars
facility_points["dv"] = facility_points["dv"].map(
    lambda x: x.name.replace("_", "").replace("x", "y")
)
facility_points

---------------------------------

## LSCP & LSCP-B with selection of predefined candidate facilities
However, in many real world applications there may already be existing facility locations with the goal being to add one or more new facilities. Here we will define facilites $y_{11}$ and $y_{15}$ as already existing (they must be present in the model solution). This will lead to a sub-optimal solution.

***Important:*** The facilities in `"predefined_loc"` are a binary array where `1` means the associated location must appear in the solution.

In [ ]:
facility_points["predefined_loc"] = 0
facility_points.loc[(11, 15), "predefined_loc"] = 1
facility_points

In [ ]:
lscp_pre = LSCP.from_cost_matrix(
    cost_matrix,
    SERVICE_RADIUS,
    predefined_facilities_arr=facility_points["predefined_loc"].values,
    name="lscp-predefined",
)
lscp_pre = lscp_pre.solve(solver)

In [ ]:
lscpb_pre = LSCPB.from_cost_matrix(
    cost_matrix,
    SERVICE_RADIUS,
    solver=solver,
    predefined_facilities_arr=facility_points["predefined_loc"].values,
    name="lscp-b-predefined",
)
lscpb_pre = lscpb_pre.solve()

In [ ]:
lscp_obj = lscp_pre.problem.objective.value()
lscpb_obj = lscpb_pre.problem.objective.value()
lscpb_lscp = lscpb_pre.lscp_obj_value
lscpb_perc = round(lscpb_pre.backup_perc, 3)
print(
    "The minimum number of facilites needed to achieve complete "
    f"coverage within a distance units service radius of {SERVICE_RADIUS} "
    f"are {lscp_obj} and {lscpb_lscp} for the LSCP and LSCP-B, "
    "respectively. However, with the LSCP-B we observe coverage by more "
    f"than one facility at {lscpb_perc}% of client locations."
)

## Plotting the results

In [ ]:
dv_colors_arr = [
    "darkcyan",
    "mediumseagreen",
    "saddlebrown",
    "darkslategray",
    "lightskyblue",
    "thistle",
    "lavender",
    "darkgoldenrod",
    "peachpuff",
    "coral",
    "mediumvioletred",
    "blueviolet",
    "fuchsia",
    "cyan",
    "limegreen",
    "mediumorchid",
]
dv_colors = {f"y{i}": dv_colors_arr[i] for i in range(len(dv_colors_arr))}
dv_colors

In [ ]:
def plot_results(model, p, facs, clis=None, ax=None):
    """Visualize optimal solution sets and context."""
    if not ax:
        multi_plot = False
        fig, ax = plt.subplots(figsize=(6, 9))
        markersize, markersize_factor = 4, 4
    else:
        ax.axis("off")
        multi_plot = True
        markersize, markersize_factor = 2, 2
    ax.set_title(model.name, fontsize=15)

    # extract facility-client relationships for plotting (except for p-dispersion)
    plot_clis = isinstance(clis, geopandas.GeoDataFrame)
    if plot_clis:
        cli_points = {}
    fac_sites = {}
    for i, dv in enumerate(model.fac_vars):
        if dv.varValue:
            dv, predef = facs.loc[i, ["dv", "predefined_loc"]]
            fac_sites[dv] = [i, predef]
            if plot_clis:
                geom = clis.iloc[model.fac2cli[i]]["geometry"]
                cli_points[dv] = geom

    # study area and legend entries initialization
    study_area.plot(ax=ax, alpha=0.5, fc="tan", ec="k", zorder=1)
    _patch = Patch(alpha=0.5, fc="tan", ec="k", label="Dissolved Service Areas")
    legend_elements = [_patch]

    if plot_clis and model.name.startswith("mclp"):
        # any clients that not asscociated with a facility
        c = "k"
        if model.n_cli_uncov:
            idx = [i for i, v in enumerate(model.cli2fac) if len(v) == 0]
            pnt_kws = {
                "ax": ax,
                "fc": c,
                "ec": c,
                "marker": "s",
                "markersize": 7,
                "zorder": 2,
            }
            clis.iloc[idx].plot(**pnt_kws)
        _label = f"Demand sites not covered ($n$={model.n_cli_uncov})"
        _mkws = {
            "marker": "s",
            "markerfacecolor": c,
            "markeredgecolor": c,
            "linewidth": 0,
        }
        legend_elements.append(mlines.Line2D([], [], ms=3, label=_label, **_mkws))

    # all candidate facilities
    facs.plot(ax=ax, fc="brown", marker="*", markersize=80, zorder=8)
    _label = f"Facility sites ($n$={len(model.fac_vars)})"
    _mkws = {"marker": "*", "markerfacecolor": "brown", "markeredgecolor": "brown"}
    legend_elements.append(mlines.Line2D([], [], ms=7, lw=0, label=_label, **_mkws))

    # facility-(client) symbology and legend entries
    zorder = 4
    for fname, (fac, predef) in fac_sites.items():
        cset = dv_colors[fname]
        if plot_clis:
            # clients
            geoms = cli_points[fname]
            gdf = geopandas.GeoDataFrame(geoms)
            gdf.plot(ax=ax, zorder=zorder, ec="k", fc=cset, markersize=100 * markersize)
            _label = f"Demand sites covered by {fname}"
            _mkws = {
                "markerfacecolor": cset,
                "markeredgecolor": "k",
                "ms": markersize + 7,
            }
            legend_elements.append(
                mlines.Line2D([], [], marker="o", lw=0, label=_label, **_mkws)
            )
        # facilities
        ec = "k"
        lw = 2
        predef_label = "predefined"
        if model.name.endswith(predef_label) and predef:
            ec = "r"
            lw = 3
            fname += f" ({predef_label})"
        facs.iloc[[fac]].plot(
            ax=ax, marker="*", markersize=1000, zorder=9, fc=cset, ec=ec, lw=lw
        )
        _mkws = {"markerfacecolor": cset, "markeredgecolor": ec, "markeredgewidth": lw}
        legend_elements.append(
            mlines.Line2D([], [], marker="*", ms=20, lw=0, label=fname, **_mkws)
        )
        # increment zorder up and markersize down for stacked client symbology
        zorder += 1
        if plot_clis:
            markersize -= markersize_factor / p

    if not multi_plot:
        # legend
        kws = {"loc": "upper left", "bbox_to_anchor": (1.05, 0.7)}
        plt.legend(handles=legend_elements, **kws)

-------------------------

## LSCP considering all candidate facilities

In [ ]:
plot_results(lscp, lscp_obj, facility_points, clis=demand_points)

## LSCP-B considering all candidate facilities

In [ ]:
plot_results(lscpb, lscpb.lscp_obj_value, facility_points, clis=demand_points)

In [ ]:
fig, axarr = plt.subplots(1, 2, figsize=(12, 6))
fig.subplots_adjust(wspace=-0.25)
for i, m in enumerate([lscp, lscpb]):
    _p = (
        m.lscp_obj_value if m.name.startswith("lscp-b") else m.problem.objective.value()
    )
    plot_results(m, _p, facility_points, clis=demand_points, ax=axarr[i])

In both the LSCP and LSCP-B solutions facility $y_1$ is included. However, $y_4$ and $y_{10}$ in the LSCP solution are swapped out for $y_3$ and $y_5$ in the LSCP-B solution to maximize backup coverage.

---------

## LSCP with selection of predefined candidate facilities

In [ ]:
plot_results(
    lscp_pre,
    lscp_pre.problem.objective.value(),
    facility_points,
    clis=demand_points,
)

## LSCP-B with selection of predefined candidate facilities

In [ ]:
plot_results(lscpb_pre, lscpb_pre.lscp_obj_value, facility_points, clis=demand_points)

-----------------------------------

## Comparing all models

In [ ]:
fig, axarr = plt.subplots(1, 2, figsize=(12, 6))
fig.subplots_adjust(wspace=-0.25)
for i, m in enumerate([lscp_pre, lscpb_pre]):
    _p = (
        m.lscp_obj_value if m.name.startswith("lscp-b") else m.problem.objective.value()
    )
    plot_results(m, _p, facility_points, clis=demand_points, ax=axarr[i])

When stipulating that $y_{11}$ and $y_{15}$ must be included in the solution, the minimum number of facilities increases to 4, but as a result backup coverage also increases. Here we can see that facility $y_1$ is again present in both solutions, where the LSCP opts for $y_6$ and the LSCP-B opts for $y_3$.

----------------------------------------

## Evaluating available solvers

First we'll determine which solvers are installed locally.

In [ ]:
with warnings.catch_warnings(record=True) as w:
    solvers = pulp.listSolvers(onlyAvailable=True)
for _w in w:
    print(_w.message)
solvers

Above we can see that it returns a list with different solvers that are available. So, it's up to the user to choose the best solver that fits the model. Let's get the percentage as a metric to evaluate which solver is the best or improves the model.

In [ ]:
results = pandas.DataFrame(
    columns=["Solution Facilities", "Backup %", "Solve Time (sec.)"], index=solvers
)
for solver in solvers:
    _solver = pulp.getSolver(solver, msg=False)
    lscpb = LSCPB.from_cost_matrix(cost_matrix, SERVICE_RADIUS, _solver).solve()
    lscpb_perc = round(
        (lscpb.problem.objective.value() / len(lscpb.cli_vars)) * 100.0, 3
    )
    results.loc[solver] = [lscpb.lscp_obj_value, lscpb_perc, lscpb.problem.solutionTime]
results